# Version 2 Dictionary Review

This notebook presents the blind adjudication packet for the 11-category Version 2 instrument. Herbal/tea has been excluded from the primary construct design, and the former sherry/rancio construct has been replaced with sherry influence: direct sherry-cask references plus explicitly adjudicated berry or dried-fruit association candidates. It intentionally displays term frequencies, concordance evidence, and approval completeness only. Scores, model results, validation contrasts, and embedding results must not be examined before the instrument is frozen.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

ROOT = Path('..') if Path.cwd().name == 'notebooks' else Path('.')
DATA = ROOT / 'data'
adjudication = pd.read_csv(DATA / 'dictionary_v2_adjudication.csv', keep_default_na=False)
ambiguous = pd.read_csv(DATA / 'dictionary_v2_ambiguous_terms.csv', keep_default_na=False)
exclusions = pd.read_csv(DATA / 'dictionary_v2_exclusions.csv', keep_default_na=False)
display(Markdown(f'**Candidate terms:** {len(adjudication):,}  |  **Registered ambiguous candidates:** {len(ambiguous):,}'))

**Candidate terms:** 322  |  **Registered ambiguous candidates:** 65

## Candidate Composition

The counts below describe the proposed construct assignment before manual approval. They are not primary measurement results.

In [2]:
composition = (adjudication.groupby(['proposed_v2_category', 'candidate_source'])
               .size().unstack(fill_value=0).assign(total=lambda x: x.sum(axis=1)))
display(composition)

candidate_source,corpus_expansion,user_proposed,v1,total
proposed_v2_category,,,,
explicit_evaluation,0,0,20,20
complexity_balance,0,0,33,33
flaws_off_notes,0,0,27,27
floral,5,1,4,10
fruit,0,0,26,26
mineral_earth_farmy,0,0,37,37
oak_cask_wood,0,0,23,23
peat_smoke_coastal,0,4,36,40
sherry_influence,0,23,39,62


## Approval Completeness Gate

All candidate terms, including ambiguity-flagged terms, require a manual decision, rationale, and approved status before `python -m analysis.dictionary freeze` can create the authoritative dictionary.

In [3]:
missing_decision = adjudication['decision'].eq('').sum()
missing_rationale = adjudication['reviewer_rationale'].eq('').sum()
pending_status = ~adjudication['reviewer_status'].eq('approved')
display(pd.DataFrame({'gate': ['Missing decisions', 'Missing rationales', 'Pending statuses'],
                      'rows': [missing_decision, missing_rationale, int(pending_status.sum())]}))
status = 'READY FOR FREEZE' if not (missing_decision or missing_rationale or pending_status.any()) else 'PENDING HUMAN REVIEW'
display(Markdown(f'**Instrument status: {status}**'))

,gate,rows
0,Missing decisions,322
1,Missing rationales,322
2,Pending statuses,322


**Instrument status: PENDING HUMAN REVIEW**

## Ambiguous-Term Register

These terms are excluded from primary rates by protocol or awaiting explicit adjudication. Alternate assignments are reserved for labeled sensitivity analysis.

In [4]:
display(ambiguous[['term', 'proposed_v2_category', 'review_frequency', 'alternate_categories', 'decision', 'reviewer_status']])
display(Markdown('**Pre-review exclusions**'))
display(exclusions)

,term,proposed_v2_category,review_frequency,alternate_categories,decision,reviewer_status
0,honey,fruit,2485,fruit;exclude_primary,,
1,jam,fruit,829,fruit;exclude_primary,,
2,marmalade,fruit,1037,fruit;exclude_primary,,
3,antiseptic,peat_smoke_coastal,218,peat_smoke_coastal;flaws_off_notes,,
4,bandage,peat_smoke_coastal,330,peat_smoke_coastal;flaws_off_notes,,
...,...,...,...,...,...,...
60,walnut,sherry_influence,1897,sherry_influence;exclude_primary,,
61,barnyard,mineral_earth_farmy,15,mineral_earth_farmy;flaws_off_notes,,
62,farmyard,mineral_earth_farmy,177,mineral_earth_farmy;flaws_off_notes,,
63,manure,mineral_earth_farmy,69,mineral_earth_farmy;flaws_off_notes,,


**Pre-review exclusions**

,term,proposed_v2_category,review_frequency,decision,reviewer_rationale,reviewer_status
0,mint,excluded_by_construct_design,1454,exclude_irrelevant,Herbal/tea is not retained as a primary constr...,design_decision
1,herbal,excluded_by_construct_design,1739,exclude_irrelevant,Herbal/tea is not retained as a primary constr...,design_decision
2,black_tea,excluded_by_construct_design,332,exclude_irrelevant,Herbal/tea is not retained as a primary constr...,design_decision
3,green_tea,excluded_by_construct_design,631,exclude_irrelevant,Herbal/tea is not retained as a primary constr...,design_decision
4,earl_grey,excluded_by_construct_design,212,exclude_irrelevant,Herbal/tea is not retained as a primary constr...,design_decision


## Review Worksheet Preview

The CSV is the editable approval artifact. The preview below keeps concordance context visible without introducing analytical outcomes.

In [5]:
columns = ['term', 'prior_v1_category', 'proposed_v2_category', 'review_frequency', 'ambiguity_flag', 'concordance_1', 'decision', 'reviewer_rationale', 'reviewer_status']
display(adjudication[columns].head(25))

,term,prior_v1_category,proposed_v2_category,review_frequency,ambiguity_flag,concordance_1,decision,reviewer_rationale,reviewer_status
0,apple,fruit_aromatics,fruit,3015,False,lour: straw. nose: it seems that it's another ...,,,
1,apricot,fruit_aromatics,fruit,574,False,en in youngish naked one such as this baby. wi...,,,
2,banana,fruit_aromatics,fruit,1182,False,"quorice and… yeah, chestnut honey. with water:...",,,
3,cherry,fruit_aromatics,fruit,690,False,"nutmeg, grass… get drier and drier in fact. wi...",,,
4,citrus,fruit_aromatics,fruit,1347,False,"ied citron, lime and ginger tonic. comment: an...",,,
5,citrus_fruit,fruit_aromatics,fruit,230,False,"deed, hint of wax, putty and various oil in th...",,,
6,grapefruit,fruit_aromatics,fruit,1547,False,w material when tasting a 17yo malt whisky. wi...,,,
7,honey,fruit_aromatics,fruit,2485,True,"/gingery side at first sip, followed by apple ...",,,
8,jam,fruit_aromatics,fruit,829,True,"and oily. great note of herb, mint, parsley, t...",,,
9,lemon,fruit_aromatics,fruit,3498,False,chive? the whole is really very fresh. mouth: ...,,,
